# 04. Kafka Deep Dive - 분산 스트리밍 플랫폼

> **난이도**: 중급 | **예상 소요**: 35분 | **사전 지식**: [01. 프로젝트 개요](./01_project_overview.ipynb) 완료

---

## [목차]
1. [환경 설정](#1.-환경-설정)
2. [Kafka 아키텍처 개요](#2.-Kafka-아키텍처-개요) — KRaft, Partition, Offset, Consumer Group
3. [Producer 설정 분석](#3.-Producer-설정-상세-분석) — linger_ms, batch_size, compression
4. [Pattern 1: Basic Produce](#4.-Pattern-1:-Basic-Produce) — send_and_wait, 자동 파티셔닝
5. [Pattern 2: Keyed Produce](#5.-Pattern-2:-Keyed-Produce) — 키 해싱, 파티션 일관성
6. [Pattern 3: Batch Produce](#6.-Pattern-3:-Batch-Produce) — send + flush, 대량 발행
7. [Topic Management](#7.-Topic-Management) — Admin Client, 토픽 CRUD
8. [Redis Stream 비교](#8.-Redis-Stream과-Kafka-비교)
9. [Kafka UI 활용법](#9.-Kafka-UI-활용법)
10. [정리](#10.-정리-및-요약)

---

## [학습 목표]
- Kafka의 Partition-Offset-Consumer Group 아키텍처 이해
- linger_ms, batch_size, compression이 성능에 미치는 영향 파악
- Key 기반 파티셔닝으로 메시지 순서를 보장하는 방법 실습

> **[주의]**: Kafka는 파티션 수를 **늘릴 수는 있지만 줄일 수는 없습니다**. 토픽 생성 시 파티션 수를 신중하게 결정하세요.

> **[관련 노트북]**: Idempotent Producer, Transactional Producer, DLT는 [07. 고급 패턴](./07_advanced_patterns.ipynb)과 [08. 안정성](./08_reliability_patterns.ipynb)에서 다룹니다.

---

### 노트북 네비게이션
| 이전 | 현재 | 다음 |
|------|------|------|
| [03. RabbitMQ Deep Dive](./03_rabbitmq_deep_dive.ipynb) | **04. Kafka Deep Dive** [현재] | [05. 벤치마크 & 시각화](./05_benchmark_and_visualization.ipynb) |

| 순서 | 주제 | 핵심 개념 |
|------|------|----------|
| 1 | 환경 설정 | httpx 비동기 클라이언트 |
| 2 | Kafka 아키텍처 개요 | KRaft, Partition, Offset, Consumer Group |
| 3 | Producer 설정 분석 | linger_ms, batch_size, compression_type |
| 4 | Basic Produce | send_and_wait, 자동 파티셔닝 |
| 5 | Keyed Produce | 키 해싱, 파티션 일관성 |
| 6 | Batch Produce | send + flush, 대량 발행 |
| 7 | Topic Management | Admin Client, 토픽 CRUD |
| 8 | Redis Stream 비교 | 개념 비교 |
| 9 | Kafka UI 활용 | http://localhost:8080 |
| 10 | 정리 | 요약 테이블 |

---
## 1. 환경 설정

프로젝트 루트를 `sys.path`에 추가하고, FastAPI 서버와 통신할 `httpx.AsyncClient`를 생성합니다.  
이후 모든 셀에서 `client` 객체를 재사용합니다.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = str(Path.cwd().parent)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import httpx
BASE_URL = "http://localhost:8000"
client = httpx.AsyncClient(base_url=BASE_URL, timeout=10.0)

서버 상태를 확인합니다. 200 응답이 돌아오면 정상입니다.

In [ ]:
resp = await client.get("/health")
print(f"Status: {resp.status_code}")
resp.json()

---
## 2. Kafka 아키텍처 개요

### 왜 Kafka의 구조를 알아야 할까?

Kafka를 사용하다 보면 "파티션을 몇 개로 해야 하지?", "왜 메시지 순서가 뒤바뀌지?", "Consumer를 더 추가했는데 왜 안 빨라지지?" 같은 질문이 생깁니다. 이런 문제를 이해하려면 Kafka의 내부 구조를 먼저 알아야 합니다.

### 대형 물류 창고로 이해하는 Kafka

Kafka를 **대형 물류 창고**에 비유하면 각 개념이 직관적으로 이해됩니다.

```text
  Kafka = 대형 물류 창고 시스템
  ========================================================

  [창고 건물 = Broker]
  ┌──────────────────────────────────────────────────────┐
  │                                                      │
  │  상품 카테고리 = Topic (예: "전자제품", "식품")          │
  │                                                      │
  │  "전자제품" 카테고리의 선반들:                           │
  │  ┌──────────────────────────────────────┐             │
  │  │ 선반 줄 0 (Partition 0)              │             │
  │  │  [1번] [2번] [3번] [4번] [5번] →     │             │
  │  ├──────────────────────────────────────┤             │
  │  │ 선반 줄 1 (Partition 1)              │             │
  │  │  [1번] [2번] [3번] →                 │             │
  │  ├──────────────────────────────────────┤             │
  │  │ 선반 줄 2 (Partition 2)              │             │
  │  │  [1번] [2번] [3번] [4번] →           │             │
  │  └──────────────────────────────────────┘             │
  │        ↑                                              │
  │    상품 번호 = Offset (순서대로 붙는 일련번호)           │
  │                                                      │
  └──────────────────────────────────────────────────────┘

  작업 팀 = Consumer Group
  ┌─────────────┐
  │ 팀원 A → 선반 줄 0 담당                               │
  │ 팀원 B → 선반 줄 1 담당                               │
  │ 팀원 C → 선반 줄 2 담당                               │
  └─────────────┘
```

| 물류 창고 비유 | Kafka 개념 | 설명 |
|---------------|-----------|------|
| 창고 건물 | **Broker** | Kafka 서버 1대. 여러 건물(Broker)이 모여 클러스터를 구성 |
| 상품 카테고리 | **Topic** | 메시지를 종류별로 분류하는 단위 (예: "주문", "결제") |
| 카테고리별 선반 줄 | **Partition** | 하나의 Topic을 여러 줄로 나눠 동시에 정리 가능하게 함 |
| 선반 위 상품 번호 | **Offset** | 각 선반 줄 안에서 상품(메시지)에 붙는 순서 번호 |
| 작업 팀 | **Consumer Group** | 같은 팀의 팀원들이 각자 다른 선반 줄을 담당 |

### Offset = 책의 페이지 번호

Offset은 **책의 페이지 번호**처럼 이해하면 쉽습니다. 책을 읽다가 어디까지 읽었는지 기억하듯, Consumer도 Offset으로 "어디까지 처리했는지"를 기억합니다.

```text
  책(Partition)의 페이지 번호(Offset):

  ┌─────┬─────┬─────┬─────┬─────┬─────┬─────┐
  │ p.0 │ p.1 │ p.2 │ p.3 │ p.4 │ p.5 │ p.6 │  ...계속 추가됨
  └─────┴─────┴─────┴─────┴─────┴─────┴─────┘
                        ↑
                   "나는 여기까지 읽었어"
                   (Consumer의 현재 Offset = 3)
                   
                   → 다음에 읽을 페이지 = p.4
                   → 이미 읽은 페이지도 다시 펼 수 있음 (재처리 가능!)
```

- 페이지 번호는 0부터 시작하고, **절대 뒤로 돌아가지 않습니다** (항상 증가)
- 한 번 붙은 번호는 바뀌지 않습니다
- "다시 읽고 싶으면" Offset을 이전 값으로 되돌리면 됩니다 (Kafka의 강력한 장점!)

### Consumer Group과 파티션의 관계 (핵심!)

> **파티션이 3개면 최대 3명이 동시에 일할 수 있습니다. 4명이 오면 1명은 놀게 됩니다.**

이것은 Kafka의 가장 중요한 규칙 중 하나입니다:

```text
  [상황 1] 팀원 3명, 선반 줄 3개 → 완벽한 배분!
  
    팀원 A ───── 선반 줄 0   (일하는 중)
    팀원 B ───── 선반 줄 1   (일하는 중)
    팀원 C ───── 선반 줄 2   (일하는 중)


  [상황 2] 팀원 4명, 선반 줄 3개 → 1명이 남는다!
  
    팀원 A ───── 선반 줄 0   (일하는 중)
    팀원 B ───── 선반 줄 1   (일하는 중)
    팀원 C ───── 선반 줄 2   (일하는 중)
    팀원 D ───── ???          (할 일이 없음... 대기 중)
    
    → Consumer를 아무리 많이 추가해도 파티션 수 이상으로는 병렬화 불가!
    → 따라서 파티션 수를 처음에 잘 결정하는 것이 중요합니다.
```

### KRaft 모드 — Kafka 4.0에서 ZooKeeper 완전 제거

Kafka 3.x까지는 **ZooKeeper**라는 별도 시스템이 Broker 상태, 파티션 리더 선출 등을 관리했습니다.
Kafka 3.3에서 KRaft가 도입되었고, **Kafka 4.0에서 ZooKeeper 코드가 완전히 삭제**되었습니다.

> **이 프로젝트는 Kafka 4.0 (KRaft 전용)을 사용합니다.**
> `docker-compose.yml`에 ZooKeeper 서비스가 없는 이유입니다.

```text
  [예전 방식]                    [현재 방식 - KRaft]
  
  Zookeeper ← 관리 → Kafka      Kafka가 스스로 관리!
  (별도 설치 필요)                (Zookeeper 불필요)
  (장애 포인트 추가)              (더 간단하고 안정적)
```

```text
  ┌─────────────────────────────────────────────┐
  │              Kafka Cluster (KRaft)           │
  │                                             │
  │  ┌─────────┐  ┌─────────┐  ┌─────────┐    │
  │  │Broker 1 │  │Broker 2 │  │Broker 3 │    │
  │  │(Controller)│         │  │         │    │
  │  └─────────┘  └─────────┘  └─────────┘    │
  │       │            │            │           │
  │       ▼            ▼            ▼           │
  │  ┌─────────────────────────────────────┐    │
  │  │         __cluster_metadata           │    │
  │  │  (Raft 합의로 메타데이터 관리)        │    │
  │  └─────────────────────────────────────┘    │
  └─────────────────────────────────────────────┘
```

---
## 3. Producer 설정 상세 분석

### 왜 Producer 설정이 중요할까?

Kafka에 메시지를 보낼 때, 메시지 하나하나를 따로 보내면 네트워크를 너무 많이 사용하게 됩니다. 마치 택배를 보낼 때 상자 1개마다 트럭을 따로 보내는 것과 같습니다. 이런 비효율을 줄이기 위해 Producer에는 **"모아서 보내기"** 설정이 있습니다.

### 택배 트럭으로 이해하는 Producer 설정

프로젝트의 `KafkaBroker.connect_producer()`는 다음 설정을 사용합니다:

```python
AIOKafkaProducer(
    linger_ms=5,
    batch_size=16384,
    compression_type="gzip",
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
    key_serializer=lambda k: k.encode("utf-8") if k else None,
)
```

이 설정을 **택배 트럭**에 비유하면 이렇습니다:

| 설정 | 택배 비유 | 실제 역할 |
|------|----------|----------|
| `linger_ms=5` | 택배를 모아서 보내는 **대기 시간** (5ms 동안 기다렸다가 한 번에 출발) | 메시지를 5ms 동안 모아서 한 번에 전송 |
| `batch_size=16384` | 트럭 **적재 한도** (16KB가 되면 대기 시간 관계없이 바로 출발) | 배치가 16KB에 도달하면 즉시 전송 |
| `compression_type="gzip"` | 택배 **압축 포장** (부피를 줄여서 트럭에 더 많이 실을 수 있음) | 배치 단위로 gzip 압축하여 네트워크 절약 |

### Before vs After: 모아서 보내기의 효과

```text
  [모아서 보내기 안 할 때] linger_ms=0
  ─────────────────────────────────────
  택배 1개 도착 → 트럭 바로 출발!      (트럭 1번 왕복)
  택배 1개 도착 → 트럭 바로 출발!      (트럭 2번 왕복)
  택배 1개 도착 → 트럭 바로 출발!      (트럭 3번 왕복)
  
  결과: 트럭 3번 왕복 = 네트워크 3번 왕복 (비효율!)


  [모아서 보낼 때] linger_ms=5
  ─────────────────────────────────────
  택배 1개 도착 → 잠깐 대기...
  택배 1개 도착 → 계속 대기...  (5ms 이내)
  택배 1개 도착 → 5ms 됐다! 트럭 출발! (택배 3개 한 번에)
  
  결과: 트럭 1번 왕복 = 네트워크 1번 왕복 (효율적!)
```

### 전송 흐름: 트럭이 출발하는 두 가지 조건

트럭(배치)은 **둘 중 하나**의 조건이 만족되면 출발합니다:
1. 대기 시간(linger_ms=5ms)이 경과했거나
2. 적재 한도(batch_size=16KB)에 도달했거나

```text
  Producer                              Broker (창고)
    │                                     │
    │── send(택배1) ──→ [트럭 적재 중]      │
    │── send(택배2) ──→ [트럭 적재 중]      │
    │                     │               │
    │      5ms 경과 또는 16KB 도달          │
    │                     │               │
    │               gzip 압축 포장         │
    │                     │               │
    │                     ├── 트럭 출발 ──→│
    │                                     │── 창고에 보관 (디스크 기록)
    │←──────── 배달 확인 (ACK) ───────────│
```

### Producer 신뢰성: acks 설정

`acks`는 메시지 발행의 **신뢰성 수준**을 결정합니다. 현재 코드에서는 기본값(`acks=1`)을 사용합니다.

| acks | 동작 | 처리량 | 데이터 안전성 |
|------|------|--------|-------------|
| `0` | ACK를 기다리지 않음 | 최고 | 최저 (유실 가능) |
| `1` (기본) | Leader 파티션 기록 후 ACK | 중간 | 중간 (Leader 장애 시 유실) |
| `all` | 모든 ISR(In-Sync Replica) 기록 후 ACK | 최저 | **최고** (안전) |

```text
Producer ──send──→ Broker (Leader)
                     │
  acks=0: 즉시 반환  │
  acks=1: ──ACK────→ │ Leader 기록 완료
  acks=all: ────────→ │ Leader + Follower 모두 기록 완료 후 ACK
```

**프로덕션 권장**: 결제, 주문 등 유실 불가 데이터는 `acks="all"` 사용.
`enable_idempotence=True`와 함께 사용하면 **정확히 한 번(exactly-once)** 전달 보장.

---
## 4. Pattern 1: Basic Produce

### 개념

가장 단순한 발행 방식입니다. **키를 지정하지 않으면** Kafka는 라운드로빈 또는 Sticky Partitioner로 파티션을 자동 선택합니다.

내부적으로 `send_and_wait()`를 사용하여 **브로커의 ACK를 기다린 후** 결과를 반환합니다.

```python
# KafkaBroker.publish() 핵심 로직
result = await self.producer.send_and_wait(destination, payload)
# result.partition → 메시지가 저장된 파티션 번호
# result.offset    → 해당 파티션 내 순번
```

**엔드포인트:** `POST /kafka/basic/publish`

In [ ]:
# Basic Produce: 단일 메시지 발행
resp = await client.post("/kafka/basic/publish", json={
    "content": "Hello Kafka!",
    "metadata": {"source": "notebook"}
})
result = resp.json()
print(f"Status: {resp.status_code}")
result

### 결과 분석

응답의 `partition`과 `offset` 필드를 확인합니다.  
여러 번 실행하면 **파티션이 변할 수 있고**, offset은 **항상 증가**합니다.

In [ ]:
# 3회 연속 발행하여 partition/offset 변화 관찰
for i in range(3):
    resp = await client.post("/kafka/basic/publish", json={
        "content": f"Message #{i}",
        "metadata": {"seq": i}
    })
    r = resp.json()
    print(f"[{i}] partition={r['partition']}  offset={r['offset']}")

위 결과에서 확인할 수 있는 것:
- **partition 값**이 0, 1, 2 중 하나로 배정됩니다 (test-topic이 3개 파티션일 경우)
- **offset 값**은 해당 파티션 내에서 단조 증가합니다
- 키가 없으므로 파티션 배정은 Sticky Partitioner에 의해 결정됩니다

---
## 5. Pattern 2: Keyed Produce

### 왜 키(Key)가 필요할까?

실제 시스템에서는 메시지의 **처리 순서**가 중요한 경우가 많습니다. 예를 들어:

> 사용자 A가 "상품 주문 -> 결제 -> 배송 시작" 순서로 이벤트를 발생시켰는데,
> 이걸 "배송 시작 -> 상품 주문 -> 결제" 순서로 처리하면 어떻게 될까요?
> 
> 아직 주문도 안 했는데 배송이 시작되는 심각한 문제가 발생합니다!

Kafka는 **같은 파티션 안에서만** 순서를 보장합니다. 키를 지정하지 않으면 메시지가 여러 파티션에 흩어져 순서가 뒤섞일 수 있습니다. 키를 지정하면 **같은 키의 메시지는 항상 같은 파티션**에 들어가므로 순서가 보장됩니다.

### 아파트 우편함으로 이해하는 Key 기반 파티셔닝

아파트 우편함을 생각해 보세요:

```text
  아파트 우편함 시스템
  ========================================
  
  편지에 적힌 동호수(Key)    우편함(Partition)
  
  "101동 201호" ──hash──→  우편함 1번 (Partition 1)
  "102동 301호" ──hash──→  우편함 0번 (Partition 0)
  "101동 201호" ──hash──→  우편함 1번 (Partition 1)  ← 같은 동호수!
  "103동 102호" ──hash──→  우편함 2번 (Partition 2)
  "101동 201호" ──hash──→  우편함 1번 (Partition 1)  ← 또 같은 동호수!
```

핵심 원리: **같은 동호수(Key)의 편지는 항상 같은 우편함(Partition)에 들어갑니다.**

이것이 가능한 이유는 Kafka가 `hash(key) % num_partitions` 공식으로 파티션을 결정하기 때문입니다. 같은 키를 넣으면 항상 같은 해시값이 나오고, 같은 파티션에 배정됩니다.

```text
  왜 이게 순서 보장에 중요할까?
  
  사용자 "user-A"의 이벤트 3개:
    1. 상품 주문
    2. 결제 완료
    3. 배송 시작
  
  key="user-A"를 지정하면:
    → 3개 이벤트 모두 Partition 1에 저장
    → Partition 1을 담당하는 Consumer 1명이 순서대로 처리
    → 주문 → 결제 → 배송 순서가 보장됨!
  
  key를 지정하지 않으면:
    → 이벤트가 Partition 0, 1, 2에 흩어짐
    → 3명의 Consumer가 각각 처리
    → 배송이 주문보다 먼저 처리될 수 있음! (문제 발생)
```

**엔드포인트:** `POST /kafka/keyed/publish`

In [ ]:
# Keyed Produce: 키 기반 파티셔닝
resp = await client.post("/kafka/keyed/publish", json={
    "key": "user-123",
    "content": "User 123 login event",
    "metadata": {"event": "login"}
})
result = resp.json()
print(f"key={result.get('key')}  partition={result['partition']}")
result

### 파티션 일관성 확인

같은 키(`user-123`)로 여러 번 발행하면 **항상 동일한 파티션**에 할당되는지 확인합니다.  
다른 키(`user-456`)는 **다른 파티션**에 할당될 수 있습니다.

In [ ]:
# 같은 키로 5회 발행 → 파티션 일관성 확인
partitions_for_key = []
for i in range(5):
    resp = await client.post("/kafka/keyed/publish", json={
        "key": "user-123",
        "content": f"Event #{i} for user-123",
        "metadata": {}
    })
    r = resp.json()
    partitions_for_key.append(r["partition"])

print(f"user-123 파티션 이력: {partitions_for_key}")
print(f"모두 동일? {len(set(partitions_for_key)) == 1}")

In [ ]:
# 다른 키로 발행하여 파티션 분포 비교
keys = ["user-A", "user-B", "user-C", "order-100", "order-200"]
for key in keys:
    resp = await client.post("/kafka/keyed/publish", json={
        "key": key,
        "content": f"Event for {key}",
        "metadata": {}
    })
    r = resp.json()
    print(f"key={key:12s}  →  partition={r['partition']}")

위 결과에서 확인할 수 있는 것:
- `user-123`으로 5회 발행한 결과가 **모두 같은 파티션**이면 키 기반 파티셔닝이 정상 동작
- 서로 다른 키들은 **해시값에 따라 다른 파티션**에 분배될 수 있음
- 파티션 수가 변경되지 않는 한, 키-파티션 매핑은 **불변**

---
## 6. Pattern 3: Batch Produce

### 왜 배치(Batch) 발행이 필요할까?

메시지를 1개씩 보내고 매번 확인(ACK)을 받는 것은 확실하지만 느립니다. 대량의 메시지를 보내야 할 때는 "모아서 한 번에" 보내는 것이 훨씬 효율적입니다.

### 고속버스 vs 택시로 이해하는 Batch Produce

```text
  [택시 방식] send_and_wait()
  ──────────────────────────────────────────
  승객 1명 탑승 → 목적지 도착 → 확인       (1회 왕복)
  승객 1명 탑승 → 목적지 도착 → 확인       (2회 왕복)
  승객 1명 탑승 → 목적지 도착 → 확인       (3회 왕복)
  
  장점: 승객 1명 1명 확실하게 도착 확인 가능
  단점: 왕복 횟수가 많아서 전체 시간이 오래 걸림


  [고속버스 방식] send() + flush()
  ──────────────────────────────────────────
  승객 1명 탑승 → 대기
  승객 1명 탑승 → 대기
  승객 1명 탑승 → 대기
  ...전부 탑승 완료 → 한 번에 출발!         (1회 왕복)
  
  장점: 왕복 1번으로 끝나서 빠르고 효율적
  단점: 중간에 문제가 생기면 어떤 승객이 탑승 실패했는지 알기 어려움
```

### send_and_wait vs send + flush 비교

| 비교 항목 | send_and_wait() (택시) | send() + flush() (고속버스) |
|----------|----------------------|---------------------------|
| **동작** | 메시지 1개 보내고 ACK 대기 | 버퍼에 쌓고 한꺼번에 전송 |
| **속도** | 느림 (매번 왕복) | 빠름 (한 번에 왕복) |
| **안전성** | 높음 (건별 확인) | 중간 (일괄 확인) |
| **적합한 상황** | 건별 확인이 필요할 때 (결제 등) | 대량 처리가 필요할 때 (로그 수집 등) |

### 핵심 트레이드오프

> **안전성 vs 성능**: 택시(send_and_wait)는 안전하지만 느리고, 고속버스(send+flush)는 빠르지만 개별 확인이 어렵습니다. 여러분의 상황에 맞게 선택하세요!

```python
# 고속버스 방식의 코드 (KafkaBroker.publish_batch() 핵심 로직)
for msg in messages:
    future = await self.producer.send(topic, payload)   # 버스에 태우기만 함
    futures.append(future)

await self.producer.flush()  # 버스 출발! (버퍼의 모든 메시지를 브로커로 전송)
```

**엔드포인트:** `POST /kafka/batch/publish`

In [ ]:
# Batch Produce: 10개 메시지 일괄 발행
messages = [
    {"content": f"Batch item #{i}", "seq": i}
    for i in range(10)
]

resp = await client.post("/kafka/batch/publish", json={
    "messages": messages
})
result = resp.json()
print(f"총 발행: {result.get('total_messages')}건")
print(f"사용된 파티션: {result.get('partitions_used')}")
result

### 대량 발행 테스트

100개 메시지를 배치로 발행하고, 파티션 분포를 확인합니다.  
키를 지정하지 않으므로 메시지가 **파티션들에 고르게 분산**되어야 합니다.

In [ ]:
# 100개 메시지 배치 발행
large_batch = [
    {"content": f"Bulk #{i}", "index": i}
    for i in range(100)
]

resp = await client.post("/kafka/batch/publish", json={
    "messages": large_batch
})
result = resp.json()
print(f"총 발행: {result.get('total_messages')}건")
print(f"사용된 파티션: {result.get('partitions_used')}")

`partitions_used` 결과를 보면:
- 파티션이 여러 개 사용되었다면 → **메시지가 분산**되어 병렬 처리 가능
- 파티션이 1개만 사용되었다면 → Sticky Partitioner가 같은 배치를 하나의 파티션에 집중시킨 것

---
## 6-1. Consumer Group 분산 소비와 Rebalance

### Consumer Group의 핵심 규칙

Kafka Consumer Group에서 파티션은 **정확히 하나의 Consumer에게만** 할당됩니다.
이 규칙 때문에 Consumer 수 > Partition 수이면 **유휴 Consumer**가 발생합니다.

```text
Consumer Group "order-service" (3 consumers, 3 partitions)
  Consumer A ──── Partition 0  ✅ 활성
  Consumer B ──── Partition 1  ✅ 활성
  Consumer C ──── Partition 2  ✅ 활성

Consumer Group "order-service" (4 consumers, 3 partitions)
  Consumer A ──── Partition 0  ✅ 활성
  Consumer B ──── Partition 1  ✅ 활성
  Consumer C ──── Partition 2  ✅ 활성
  Consumer D ──── (없음)       ⚠️ 유휴 (파티션이 부족!)
```

### Rebalance (재할당)

Consumer가 **추가/제거**되면 Kafka는 자동으로 **Rebalance**를 수행합니다:
1. 모든 Consumer의 파티션 할당을 해제
2. 새로운 할당을 계산하여 재배치
3. 이 과정에서 **일시적으로 메시지 소비가 중단**됩니다

```text
[Before] Consumer A: P0, P1  |  Consumer B: P2
    ↓ Consumer C 추가
[Rebalancing...] 모든 소비 일시 중단
    ↓
[After]  Consumer A: P0  |  Consumer B: P1  |  Consumer C: P2
```

### Rebalance가 위험한 이유

| 상황 | 위험 |
|------|------|
| Rebalance 중 소비 중단 | 처리 지연 발생 |
| 처리 중인 메시지의 offset 미커밋 | 중복 처리 가능 |
| 빈번한 Consumer 재시작 | Rebalance Storm |

**프로덕션 팁**: `session.timeout.ms`와 `heartbeat.interval.ms`를 적절히 설정하고,
`max.poll.interval.ms`를 처리 시간에 맞게 조정해야 합니다.

### 파티션 수 설계 원칙

| 원칙 | 설명 |
|------|------|
| 파티션 수 ≥ 예상 Consumer 수 | 모든 Consumer에게 파티션 할당 보장 |
| 파티션 수는 늘릴 수만 있음 | 줄이면 키 기반 라우팅이 깨짐 |
| 일반적 시작값 | 3~12개 (트래픽에 따라 조정) |

---
## 7. Topic Management

### 개념

Kafka의 **Admin Client**를 사용하면 토픽을 프로그래밍 방식으로 생성하고 관리할 수 있습니다.  
`KafkaBroker`는 `AIOKafkaAdminClient`를 래핑하여 다음 기능을 제공합니다:

| 메서드 | 역할 |
|--------|------|
| `create_topic(name, partitions, replication)` | 새 토픽 생성 |
| `list_topics()` | 클러스터 내 토픽 목록 |
| `topic_info(topic)` | 특정 토픽의 파티션 상세 정보 |

In [ ]:
# 토픽 생성: notebook-demo (4 파티션)
resp = await client.post(
    "/kafka/topic/create",
    params={"topic": "notebook-demo", "partitions": 4}
)
print(f"Status: {resp.status_code}")
resp.json()

토픽이 이미 존재하면 `{"status": "already_exists"}`가 반환됩니다.  
이제 클러스터의 전체 토픽 목록을 조회합니다.

In [ ]:
# 토픽 목록 조회
resp = await client.get("/kafka/topics")
data = resp.json()

print(f"클러스터 정보: {data.get('cluster')}")
print(f"일반 토픽: {data.get('topics')}")
print(f"내부 토픽: {data.get('internal_topics')}")

`internal_topics`에 `__consumer_offsets`가 보입니다.  
이것은 Kafka가 Consumer Group의 offset을 추적하기 위해 자동 생성하는 내부 토픽입니다.

이제 특정 토픽의 파티션 상세 정보를 조회합니다.

In [ ]:
# 토픽 상세 정보 조회
resp = await client.get("/kafka/topic/info/test-topic")
info = resp.json()

print(f"토픽: {info.get('topic')}")
print(f"파티션 수: {info.get('num_partitions')}")
for p in info.get("partitions", []):
    print(f"  Partition {p['partition']}")

In [ ]:
# 방금 생성한 notebook-demo 토픽 정보도 확인
resp = await client.get("/kafka/topic/info/notebook-demo")
info = resp.json()

print(f"토픽: {info.get('topic')}")
print(f"파티션 수: {info.get('num_partitions')}")
print("→ 4개 파티션으로 생성한 것이 확인됩니다")

---
## 8. Redis Stream과 Kafka 비교

### 당신의 시스템에 맞는 선택은? (결정 트리)

아래 질문을 따라가면 어떤 기술이 적합한지 빠르게 판단할 수 있습니다:

```text
  질문 1: 메시지가 하루에 100만 건 이상인가?
    │
    ├── YES → Kafka (대용량 처리에 특화)
    │
    └── NO
         │
         질문 2: 이미 Redis를 사용 중이고, 간단한 이벤트 로그만 필요한가?
           │
           ├── YES → Redis Stream (추가 인프라 없이 바로 사용 가능)
           │
           └── NO
                │
                질문 3: 여러 서버에서 분산 처리가 필요한가?
                  │
                  ├── YES → Kafka (파티션 기반 분산 처리)
                  │
                  └── NO → Redis Stream (단일 서버에서 빠르고 간단하게)
```

### 상세 비교표

이 프로젝트는 Redis Stream과 Kafka를 모두 지원합니다.  
두 시스템의 핵심 차이를 정리합니다.

| 항목 | Kafka | Redis Stream |
|------|-------|--------------|
| **설계 목적** | 대용량 이벤트 스트리밍 | 경량 메시지 큐 + 캐시 |
| **저장** | 디스크 (retention 기반) | 메모리 (maxlen 기반) |
| **파티셔닝** | Topic -> Partition (키 해싱) | Stream은 단일 로그 |
| **소비 모델** | Consumer Group (파티션 할당) | Consumer Group (pending 관리) |
| **순서 보장** | 파티션 내 보장 | 전체 Stream 내 보장 |
| **처리량** | 초당 수십만~수백만 건 | 초당 수만~십만 건 |
| **재처리** | offset 이동으로 자유롭게 | XRANGE로 가능하지만 제한적 |
| **ACK 방식** | offset commit | XACK (개별 메시지) |
| **압축** | 배치 단위 gzip/snappy/lz4 | 미지원 |
| **적합 사례** | 로그 수집, 이벤트 소싱, CDC | 실시간 알림, 세션 이벤트 |

### 한 줄 요약

- **Kafka 선택**: 데이터 유실 불가, 대용량, 장기 보관, 다수 Consumer가 필요할 때
- **Redis Stream 선택**: 저지연 필수, 간단한 구조, 이미 Redis를 사용 중일 때

---
## 9. Kafka UI 활용법

이 프로젝트의 Docker Compose에는 **Kafka UI**가 포함되어 있습니다.  
브라우저에서 접속하여 클러스터 상태를 시각적으로 확인할 수 있습니다.

### 접속 주소

**http://localhost:8080**

### 주요 메뉴

| 메뉴 | 확인 가능한 정보 |
|------|------------------|
| **Dashboard** | 브로커 수, 토픽 수, 파티션 총 수 |
| **Topics** | 토픽 목록, 파티션 수, 메시지 수, 설정 |
| **Topics → Messages** | 토픽 내 실제 메시지 내용 확인 (JSON 뷰) |
| **Topics → Consumers** | 어떤 Consumer Group이 해당 토픽을 구독 중인지 |
| **Consumers** | Consumer Group별 lag (처리 지연) 확인 |
| **Brokers** | 브로커 상태, 설정, JMX 메트릭 |

### 실습에서 확인할 것

1. **Topics** 메뉴에서 `test-topic`과 `notebook-demo` 확인
2. `test-topic` 클릭 → **Messages** 탭에서 방금 발행한 메시지 확인
3. 메시지의 `partition`, `offset`, `key`, `value` 필드 확인
4. **Consumers** 메뉴에서 Consumer Group의 lag 확인

In [ ]:
# Kafka UI 접속 확인 (로컬에서만 동작)
try:
    ui_client = httpx.AsyncClient(timeout=5.0)
    ui_resp = await ui_client.get("http://localhost:8080")
    print(f"Kafka UI 상태: {ui_resp.status_code}")
    print("→ 브라우저에서 http://localhost:8080 으로 접속하세요")
    await ui_client.aclose()
except Exception as e:
    print(f"Kafka UI 접속 불가: {e}")
    print("→ docker compose up 으로 실행 중인지 확인하세요")

---
## 10. 정리 및 요약

### Producer 패턴 요약

| 패턴 | 메서드 | 내부 동작 | 용도 |
|------|--------|-----------|------|
| Basic Produce | `publish()` | `send_and_wait()` | 단건 발행, ACK 필요 |
| Keyed Produce | `publish_keyed()` | `send_and_wait(key=...)` | 순서 보장, 키 기반 라우팅 |
| Batch Produce | `publish_batch()` | `send()` × N + `flush()` | 대량 처리, 높은 처리량 |

### API 엔드포인트 요약

| 엔드포인트 | 메서드 | 기능 |
|-----------|--------|------|
| `/kafka/basic/publish` | POST | Basic Produce |
| `/kafka/keyed/publish` | POST | Keyed Produce |
| `/kafka/batch/publish` | POST | Batch Produce |
| `/kafka/topic/create` | POST | 토픽 생성 |
| `/kafka/topics` | GET | 토픽 목록 조회 |
| `/kafka/topic/info/{topic}` | GET | 토픽 상세 정보 |

### 핵심 개념 요약

| 개념 | 설명 |
|------|------|
| **Partition** | 토픽의 병렬 처리 단위. 파티션 수 = 최대 병렬 Consumer 수 |
| **Offset** | 파티션 내 메시지 순번. Consumer는 이 값으로 읽기 위치 추적 |
| **Consumer Group** | 같은 group_id의 Consumer가 파티션을 분배받아 병렬 처리 |
| **Key Hashing** | 키의 해시값으로 파티션 결정. 같은 키 → 같은 파티션 보장 |
| **linger_ms** | 메시지를 모아서 보내는 대기 시간 (처리량 ↔ 지연 트레이드오프) |
| **batch_size** | 배치 최대 크기. 도달하면 linger_ms와 무관하게 즉시 전송 |
| **compression** | 배치 단위 압축. 네트워크 절약, CPU 비용 증가 |

### Kafka 4.0 — 무엇이 달라졌나?

이 프로젝트는 **Kafka 4.0**을 사용합니다. 3.x 대비 주요 변경사항:

| 변경사항 | 설명 |
|----------|------|
| **ZooKeeper 완전 제거** | ZooKeeper 관련 코드가 삭제됨. KRaft가 유일한 메타데이터 관리 방식 |
| **KRaft 전용** | `KAFKA_PROCESS_ROLES: broker,controller` — 하나의 프로세스가 두 역할 수행 |
| **Tiered Storage GA** | 핫 데이터는 로컬, 콜드 데이터는 S3 등 오브젝트 스토리지에 자동 계층화 |
| **Consumer Group Protocol v2** | 리밸런싱 시간 대폭 단축 (Server-side 할당) |

> **왜 중요한가?** — ZooKeeper 제거로 운영 복잡도가 크게 줄었습니다.
> 이전에는 Kafka + ZooKeeper 두 시스템을 모두 모니터링하고 관리해야 했지만,
> 이제 Kafka 클러스터만 관리하면 됩니다.

In [ ]:
# 클라이언트 정리
await client.aclose()
print("httpx 클라이언트 종료 완료")